In [ ]:
#importing the library
import os
import pandas as pd
import numpy as np
import re
import string
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import cohen_kappa_score, accuracy_score
from transformers import BertTokenizer, BertModel
import torch
from sklearn.base import BaseEstimator, TransformerMixin
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
essays = []
for i in range(1, 56):
    with open(f"d{i}.txt", "r", encoding="utf-8") as file:
        content = file.read().strip()
        essays.append(content)

scores = [4]*15 + [3]*15 + [2]*15 + [1]*10

df = pd.DataFrame({"essay": essays, "score": scores})

In [ ]:
def preprocess_words(words):
    return [
        lemmatizer.lemmatize(re.sub(r'[^\w\s]', '', word.lower()))
        for word in words
        if word and not word.startswith('@') and word.lower() not in stop_words
    ]

In [ ]:
class BertFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='bert-base-uncased', max_length=512):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name)
        self.max_length = max_length

    def transform(self, X):
        self.model.eval()  # set to evaluation mode
        features = []

        with torch.no_grad():
            for text in X:
                inputs = self.tokenizer(text, return_tensors='pt', padding='max_length',
                                        truncation=True, max_length=self.max_length)
                outputs = self.model(**inputs)
                cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # CLS token
                features.append(cls_embedding)

        return np.array(features)

    def fit(self, X, y=None):
        return self


In [ ]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def transform(self, X):
        features = []
        for essay in X:
            tokens = essay.split()
            word_count = len(tokens)
            error_count = sum(1 for w in tokens if not w.isalpha())
            error_rate = error_count / word_count if word_count else 0

            sentences = re.split(r'[.!?]', essay)
            sentences = [s for s in sentences if s.strip()]
            avg_sent_len = word_count / len(sentences) if sentences else 0

            cleaned = preprocess_words(tokens)
            top_word_count = pd.Series(cleaned).value_counts().iloc[0] if cleaned else 0

            features.append([word_count, error_rate, avg_sent_len, top_word_count])
        return np.array(features)

    def fit(self, X, y=None):
        return self

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df["essay"], df["score"], test_size=0.25, random_state=42)

In [ ]:
# Define your custom input
custom_essay = input("Enter your essay text: ")
custom_df = pd.DataFrame({"essay": [custom_essay]})

# Valid score levels
valid_scores = sorted(df["score"].unique())

# Models to evaluate
models = {
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Train each model on full data and predict the custom essay
print("\nPredicted Scores for Input Essay:\n")
for name, model in models.items():
    pipeline = Pipeline([
        ("features", BertFeatureExtractor()),
        ("model", model)
    ])
    pipeline.fit(df["essay"], df["score"])

    raw_pred = pipeline.predict(custom_df)[0]
    rounded_pred = min(valid_scores, key=lambda x: abs(x - raw_pred))

    print(f"{name:<25} => Raw Score: {raw_pred:.2f}, Rounded Score: {rounded_pred}")

Enter your essay text: Dear Local Newspaper @CAPS1, @CAPS2 our technology advance, so do are lives. Computers are a perfect way to better our way of life in ways that are benificial to us and future generations. The use of computers, and the @CAPS5 alone allow us to learn about the way of life in other places on @LOCATION4 talk to people faraway, and express ourselves. Computers give us a bright future. First many people, actually @PERCENT1 of the worlds population are generally stationary, and line in a local town or city environments. This is why computers are so helpful because they give @CAPS2 a way to research other places around the globe. @CAPS3 can create an interest in other cultures, and shape our opinions @CAPS2 @CAPS3 discover things about other places. From there, @CAPS3 can look at different topics in a wide spread point of view. In doing this @CAPS3 will connect to other ways of life. @CAPS2 philosopher and poet @PERSON1 state" @CAPS3 learn about ourselves through the di